# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Dataset attached at `/kaggle/input/datasets/wenhaolu49/cipher`.

| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35% | Plan quality vs. oracle beam search |
| Calibration | 25% | Brier score on stated self-knowledge |
| Attention | 20% | Rank correlation on unknown importance |
| Executive | 20% | Plan structure, named risks, alternative plans |

In [ ]:
# 1. Install dependencies
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks", "pandas", "matplotlib", "numpy")
print("Dependencies installed.")

In [ ]:
# 2. Imports
import os, sys, json
import pandas as pd
import kaggle_benchmarks as kbench
from typing import Any, Dict, List, Optional
from kaggle_benchmarks.prompting import ResponseParsingError

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found — attach the cipher dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect
from cipher.schema import validate_response
from cipher.scorer import score_response
print("cipher imports OK.")

In [ ]:
# 3. Load instances
DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError("data/instances.jsonl not found.")

KBENCH_LIMIT = 10

with open(DATA_PATH) as f:
    ALL_RECORDS = [json.loads(line) for line in f if line.strip()]
if KBENCH_LIMIT:
    ALL_RECORDS = ALL_RECORDS[:KBENCH_LIMIT]

print(f"Loaded {len(ALL_RECORDS)} instances from {DATA_PATH}")
print("\nAvailable models:", list(kbench.llms.keys()))

In [ ]:
# 4. Scoring helpers
def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    h   = rec["hidden"]
    hrl = {e["rule_name"]: set(e["hidden"]) for e in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t, e  = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(kind=t["kind"], i=t["i"], j=t.get("j",-1), k=t.get("k",0),
                            hidden_kind=("trigger_kind" in hides), hidden_k=("trigger_k" in hides)),
            effect=Effect(kind=e["kind"], target=e["target"],
                          delta=e.get("delta",0), source=e.get("source",-1),
                          hidden_kind=("effect_kind" in hides), hidden_delta=("effect_delta" in hides)),
        ))
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[], hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )

_ZERO = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)

def _score(raw_text: str, rec: Dict[str, Any]) -> Dict[str, float]:
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start == -1:
        return _ZERO.copy()
    try:
        raw  = json.loads(raw_text[start:end+1])
        inst = _instance_from_record(rec)
        resp = validate_response(raw)
        bd   = score_response(resp, inst,
                              best_obj=rec["hidden"].get("oracle_best"),
                              worst_obj=rec["hidden"].get("oracle_worst"))
        return bd.to_dict()
    except Exception:
        return _ZERO.copy()

In [ ]:
# 5. Per-instance task
@kbench.task(name="cipher_single_instance_scorer_v2", store_task=False)
def cipher_single_instance_scorer(llm, prompt: str, record_json: str) -> dict:
    try:
        reply = llm.prompt(prompt)
        rec   = json.loads(record_json)
        return _score(str(reply), rec)
    except (ResponseParsingError, json.JSONDecodeError, Exception):
        return _ZERO.copy()

print("cipher_single_instance_scorer defined.")

In [ ]:
# 6. Batch evaluation
_LAST_RUN = {}

@kbench.task(name="cipher_eval_full")
def cipher_metacognition_evaluation(llm):
    """CIPHER: micro-worlds with hidden causal rules in invented vocabulary. Tests metacognitive calibration."""
    evaluation_df = pd.DataFrame([{
        "prompt":      rec["prompt"],
        "record_json": json.dumps(rec),
    } for rec in ALL_RECORDS])

    with kbench.client.enable_cache():
        runs = cipher_single_instance_scorer.evaluate(
            llm=[llm],
            evaluation_data=evaluation_df,
            n_jobs=2,
            remove_run_files=True,
        )

    results_df = runs.as_dataframe()

    per_instance = []
    for rec, result in zip(ALL_RECORDS, results_df["result"]):
        row = {"id": rec["id"], "difficulty": rec["difficulty"]}
        if isinstance(result, dict):
            row.update(result)
        else:
            row.update(_ZERO)
        per_instance.append(row)

    _LAST_RUN["per_instance"] = per_instance

    def _mean(key):
        return float(sum(r.get(key, 0.0) for r in per_instance) / max(len(per_instance), 1))

    scores = {k: _mean(k) for k in ["composite", "objective", "calibration", "attention", "executive"]}
    _LAST_RUN["scores"] = scores

    print(f"composite={scores['composite']:.3f}  obj={scores['objective']:.3f}  "
          f"cal={scores['calibration']:.3f}  att={scores['attention']:.3f}  exe={scores['executive']:.3f}")
    return scores["composite"]

cipher_metacognition_evaluation.run(kbench.llm)

In [ ]:
%choose cipher_eval_full

In [ ]:
# 7. Analysis
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if not _LAST_RUN:
    print("No run data — execute cell 6 first.")
else:
    df_r   = pd.DataFrame(_LAST_RUN["per_instance"])
    scores = _LAST_RUN["scores"]
    DIMS   = ["composite", "objective", "calibration", "attention", "executive"]
    COLORS = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]

    BASELINES = {
        "stub-noop":   {"composite":0.408,"objective":0.486,"calibration":0.750,"attention":0.000,"executive":0.250},
        "stub-random": {"composite":0.511,"objective":0.484,"calibration":0.663,"attention":0.211,"executive":0.670},
        "stub-greedy": {"composite":0.623,"objective":1.000,"calibration":0.893,"attention":0.000,"executive":0.250},
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("CIPHER Benchmark Results", fontsize=14, fontweight="bold")

    ax = axes[0, 0]
    x  = np.arange(len(DIMS))
    ax.bar(x, [scores[d] for d in DIMS], color=COLORS, zorder=3, label="Model")
    ls = ["--", "-.", ":"]
    for (bname, bscores), style in zip(BASELINES.items(), ls):
        ax.plot(x, [bscores[d] for d in DIMS], style, linewidth=1.5, label=bname)
    ax.set_xticks(x); ax.set_xticklabels(DIMS, rotation=15, ha="right")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title("Scores by Dimension vs Baselines")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3, zorder=0)

    ax = axes[0, 1]
    ax.hist(df_r["composite"], bins=15, color="#2196F3", edgecolor="white", zorder=3)
    ax.axvline(scores["composite"], color="blue",   linestyle="-",  linewidth=2, label=f"Mean ({scores['composite']:.3f})")
    ax.axvline(0.623,              color="red",    linestyle="--", linewidth=1.5, label="stub-greedy (0.623)")
    ax.axvline(0.408,              color="gray",   linestyle=":",  linewidth=1.2, label="stub-noop (0.408)")
    ax.set_xlabel("Composite Score"); ax.set_ylabel("Count")
    ax.set_title("Composite Score Distribution"); ax.legend(fontsize=8); ax.grid(alpha=0.3, zorder=0)

    ax = axes[1, 0]
    diff_colors = {"easy": "#4CAF50", "medium": "#FF9800", "hard": "#F44336"}
    for diff, grp in df_r.groupby("difficulty"):
        ax.scatter(grp["objective"], grp["calibration"],
                   c=diff_colors.get(diff, "gray"), label=diff, alpha=0.7, s=40, zorder=3)
    ax.set_xlabel("Objective"); ax.set_ylabel("Calibration")
    ax.set_title("Objective vs Calibration (key metacog tradeoff)")
    ax.legend(title="Difficulty", fontsize=8); ax.grid(alpha=0.3, zorder=0)

    ax = axes[1, 1]
    diff_means = df_r.groupby("difficulty")[DIMS].mean().reindex(["easy", "medium", "hard"])
    diff_means.plot(kind="bar", ax=ax, color=COLORS, width=0.7, zorder=3)
    ax.set_xticklabels(diff_means.index, rotation=0); ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score"); ax.set_title("Scores by Difficulty")
    ax.legend(fontsize=7, ncol=2); ax.grid(axis="y", alpha=0.3, zorder=0)

    plt.tight_layout()
    plt.savefig("cipher_results.png", dpi=150, bbox_inches="tight")
    plt.close()
    sys.stdout.flush()
    print("Saved: cipher_results.png")

    print("\n=== Per-difficulty breakdown ===")
    print(diff_means.round(3).to_string())

    print("\n=== Overall scores ===")
    for k, v in scores.items():
        print(f"  {k:<12} {v:.3f}")

    print("\n=== Baseline comparison ===")
    print(f"  {'model':<14} {'composite':>10} {'objective':>10} {'calibration':>12} {'attention':>10} {'executive':>10}")
    for bname, bscores in BASELINES.items():
        print(f"  {bname:<14} {bscores['composite']:>10.3f} {bscores['objective']:>10.3f} "
              f"{bscores['calibration']:>12.3f} {bscores['attention']:>10.3f} {bscores['executive']:>10.3f}")
    print(f"  {'this model':<14} {scores['composite']:>10.3f} {scores['objective']:>10.3f} "
          f"{scores['calibration']:>12.3f} {scores['attention']:>10.3f} {scores['executive']:>10.3f}")
    sys.stdout.flush()